In [ ]:
#import libraries
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import scipy.integrate as integrate
from scipy import interpolate
from scipy.interpolate import UnivariateSpline
import pickle

import pypopsyn.simulator.basics.constants as const
from pypopsyn.simulator.config_simulator import cfg
import utilities.plot_settings

In [ ]:
# Dominant conductivity based on phonon or impurity scattering, in [1/s].
# For details see Cumming et al. (2004) or Gourgouliatos and Cumming (2014).
sigma_min = 1.8e23
sigma_max = 3.6e24
sigma = 1.e24

# Characteristic length scale of the magnetic field in [cm].
L = 1e5

# Characteristic electron density in [g/cm^3].
n_e_min = 2.5e34
n_e_max = 2.5e36
n_e = 1.e35


In [ ]:
def timescale_ohmic(L: float, sigma: float) -> float:
    """
    Calculating the ohmic diffusion timescale for a given conductivity and characteristic magnetic
    field length scale. Note that for our purposes, we neglect the fact that both quantities can
    vary significantly with depth inside the neutron star, and we simply use effective quantities
    that reflect the ohmic diffusion process. For our choices see the configuration file.

    Args:
        L (float): characteristic length scale on which the magnetic field varies, measured in [cm].
        sigma (float): conductivity of the dominating dissipative process, measure in [1/s].

    Returns:
        (float): ohmic diffusion timescale in [yr].
    """

    tau_ohm = 4 * np.pi * sigma * L ** 2 / (const.C ** 2) / const.YR_TO_S
    #tau_ohm = 3.e6 * const.YR_TO_S

    return tau_ohm


def timescale_Hall(B: float, L: float, n_e: float) -> float:
    """
    Calculating the Hall timescale for a given field strength, characteristic magnetic field length
    scale and electron density. Note that for our purposes, we neglect the fact that all quantities can
    vary significantly with depth inside the neutron star, and we simply use effective quantities
    that reflect the conservative Hall process. For our choices of L and n_e see the configuration file.
    B will be identified with the initial dipolar magnetic field components at the pulsars' pole.

    Args:
        B (float): (local) magnetic field strength, measured in [G].
        L (float): characteristic length scale on which the magnetic field varies, measured in [cm].
        n_e (float): electron density, measured in [g/cm^3].

    Returns:
        (float): Hall timescale in [yr].
    """

    tau_Hall = (
        4 * np.pi * const.E * n_e * L ** 2 / (const.C * B)
    ) / const.YR_TO_S
    #tau_Hall = 1.e4 * (1.e15/B) * const.YR_TO_S

    return tau_Hall


def Bfield_evol(t: np.ndarray, B_initial: float,) -> np.ndarray:
    """
    Calculating the change in the magnetic field strength of a pulsar based on a simplified
    differential equation (see eq. (18) of Aguilera et al. (2008)) that captures the
    characteristics of more complication numerical simulations of pulsar magnetic field
    evolution, i.e., at early timescales the Hall evolution dominates, while at late times
    the exponential magnetic field decay due to Ohmic dissipation kicks in. Note that as
    explained in Aguilera et al. (2008) the Hall timescale corresponds to that of the initial
    field strength.

    Args:
        t (np.ndarray): time array in [yr].
        B_initial (float): pulsar's initial magnetic field magnitudes, measured in [G].

    Returns:
        (np.ndarray): magnetic field as a function of time for a simulated pulsars in [G].
    """
    tau_ohm = timescale_ohmic(L, sigma)
    tau_Hall = timescale_Hall(B_initial, L, n_e)

    B_t = B_initial * np.exp(- t / tau_ohm) / (1 + tau_ohm/tau_Hall * (1 - np.exp(- t / tau_ohm)))
    
    return B_t

In [ ]:
df_B12 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e12_Btor1e13.csv",
    delimiter=",",
    header=[0],
)
df_B12.head()

In [ ]:
df_B13 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e13_Btor1e14.csv",
    delimiter=",",
    header=[0],
)
df_B13.head()

In [ ]:
df_B14 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e14_Btor1e15.csv",
    delimiter=",",
    header=[0],
)
df_B14.head()

In [ ]:
df_B15 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e15_Btor1e16.csv",
    delimiter=",",
    header=[0],
)
df_B15.head()

In [ ]:
df_B5e15 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip5e15_Btor1e16.csv",
    delimiter=",",
    header=[0],
)
df_B5e15.head()

In [ ]:
t12 = df_B12["t[yr]"].to_numpy().astype(float) 
t13 = df_B13["t[yr]"].to_numpy().astype(float) 
t14 = df_B14["t[yr]"].to_numpy().astype(float) 
t15 = df_B15["t[yr]"].to_numpy().astype(float) 
t5e15 = df_B5e15["t[yr]"].to_numpy().astype(float) 
B12 = df_B12["B[G]"].to_numpy().astype(float) 
B13 = df_B13["B[G]"].to_numpy().astype(float) 
B14 = df_B14["B[G]"].to_numpy().astype(float) 
B15 = df_B15["B[G]"].to_numpy().astype(float) 
B5e15 = df_B5e15["B[G]"].to_numpy().astype(float) 
L12 = df_B12["L[erg/s]"].to_numpy().astype(float) 
L13 = df_B13["L[erg/s]"].to_numpy().astype(float) 
L14 = df_B14["L[erg/s]"].to_numpy().astype(float) 
L15 = df_B15["L[erg/s]"].to_numpy().astype(float) 
L5e15 = df_B5e15["L[erg/s]"].to_numpy().astype(float) 

In [ ]:
log_B_range = np.array([10, 12, 13, 14, 15, np.log10(3.e15)])

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(log_B_range))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale('log') 
ax.set_yscale('log')
#ax.set_xlim(P_min,P_max) 
ax.set_ylim(1.e11,1.e16) 
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$B_{\rm dip}$ [G]')

ax.plot( 
    t12,
    B12,
    linestyle='-',
    linewidth=4,
    color=colors(1),
    rasterized=True,
)

ax.plot( 
    t13,
    B13,
    linestyle='-',
    linewidth=4,
    color=colors(2),
    rasterized=True,
)

ax.plot( 
    t14,
    B14,
    linestyle='-',
    linewidth=4,
    color=colors(3),
    rasterized=True,
)

ax.plot( 
    t15,
    B15,
    linestyle='-',
    linewidth=4,
    color=colors(4),
    rasterized=True,
)

ax.plot( 
    t5e15,
    B5e15,
    linestyle='-',
    linewidth=4,
    color=colors(5),
    rasterized=True,
)

In [ ]:
print(len(t12))
print(np.min(t12))
print(np.max(t12))
print(len(t13))
print(np.min(t13))
print(np.max(t13))
print(len(t14))
print(len(B14))
print(np.min(t14))
print(np.max(t14))
print(len(t15))
print(np.min(t15))
print(np.max(t15))
print(len(t5e15))
print(np.min(t5e15))
print(np.max(t5e15))

In [ ]:
t = np.logspace(0.,6., 100)

In [ ]:
fB12 = interpolate.UnivariateSpline(t12, B12, k=1)
fB13 = interpolate.UnivariateSpline(t13, B13, k=1)
fB14 = interpolate.UnivariateSpline(t14, B14, k=1)
fB15 = interpolate.UnivariateSpline(t15, B15, k=1)
fB5e15 = interpolate.UnivariateSpline(t5e15, B5e15, k=1)

B10_new = Bfield_evol(t, 10**(10))
B11_new = Bfield_evol(t, 10**(11))
B12_new = fB12(t)
B13_new = fB13(t)
B14_new = fB14(t)
B15_new = fB15(t)
B5e15_new = fB5e15(t)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(log_B_range))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale('log') 
ax.set_yscale('log')
#ax.set_xlim(P_min,P_max) 
ax.set_ylim(1.e9,1.e16) 
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$B_{\rm dip}$ [G]')

ax.plot( 
    t,
    B11_new,
    linestyle='-',
    linewidth=4,
    color=colors(0),
    rasterized=True,
)

ax.plot( 
    t,
    B12_new,
    linestyle='-',
    linewidth=4,
    color=colors(1),
    rasterized=True,
)

ax.plot( 
    t,
    B13_new,
    linestyle='-',
    linewidth=4,
    color=colors(2),
    rasterized=True,
)

ax.plot( 
    t,
    B14_new,
    linestyle='-',
    linewidth=4,
    color=colors(3),
    rasterized=True,
)

ax.plot( 
    t,
    B15_new,
    linestyle='-',
    linewidth=4,
    color=colors(4),
    rasterized=True,
)

ax.plot( 
    t,
    B5e15_new,
    linestyle='-',
    linewidth=4,
    color=colors(5),
    rasterized=True,
)
plt.grid()

In [ ]:
# Create a table grid of magnetic field evolution.
B0 = np.array([1.e10, 1.e11, 1.e12, 1.e13, 1.e14, 1.e15, 5.e15])

B_t = np.vstack((B10_new, B11_new, B12_new, B13_new, B14_new, B15_new, B5e15_new)).T
print(t.shape)
print(B_t.shape)

B_interpolator = interpolate.RectBivariateSpline(
    t, 
    B0, 
    B_t, 
    bbox=[0.0, 1.e8, 1.e9, 1.e17], 
    kx=1, 
    ky=1
)

t_eval = np.logspace(0., 7., 500)
B0_eval = np.logspace(9., 16, 30)

Bt_interp = B_interpolator(t_eval, B0_eval)
print(Bt_interp.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(B0_eval))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale('log') 
ax.set_yscale('log')
#ax.set_xlim(P_min,P_max) 
#ax.set_ylim(Pdot_min,Pdot_max) 
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$B_{\rm dip}$ [G]')

ax.plot( 
    t12,
    B12,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)

ax.plot( 
    t13,
    B13,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)

ax.plot( 
    t14,
    B14,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)

ax.plot( 
    t15,
    B15,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)

ax.plot( 
    t5e15,
    B5e15,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
for i in range(len(B0_eval)):
    ax.plot( 
        t_eval,
        Bt_interp[:,i],
        linestyle='-',
        linewidth=4,
        color=colors(i),
        rasterized=True,
        alpha=0.5
    )


plt.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(log_B_range))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale('log') 
ax.set_yscale('log')
#ax.set_xlim(P_min,P_max) 
#ax.set_ylim(Pdot_min,Pdot_max) 
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$L_{X}$ [erg s$^{-1}$]')

ax.plot( 
    t12,
    L12,
    linestyle='-',
    linewidth=4,
    color=colors(1),
    rasterized=True,
)
ax.plot( 
    t13,
    L13,
    linestyle='-',
    linewidth=4,
    color=colors(2),
    rasterized=True,
)
ax.plot( 
    t14,
    L14,
    linestyle='-',
    linewidth=4,
    color=colors(3),
    rasterized=True,
)
ax.plot( 
    t15,
    L15,
    linestyle='-',
    linewidth=4,
    color=colors(4),
    rasterized=True,
)
ax.plot( 
    t5e15,
    L5e15,
    linestyle='-',
    linewidth=4,
    color=colors(5),
    rasterized=True,
)

In [ ]:
fL12 = interpolate.UnivariateSpline(t12, L12, k=1)
fL13 = interpolate.UnivariateSpline(t13, L13, k=1)
fL14 = interpolate.UnivariateSpline(t14, L14, k=1)
fL15 = interpolate.UnivariateSpline(t15, L15, k=1)
fL5e15 = interpolate.UnivariateSpline(t5e15, L5e15, k=1)

L12_new = fL12(t)
L13_new = fL13(t)
L14_new = fL14(t)
L15_new = fL15(t)
L5e15_new = fL5e15(t)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(log_B_range))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale('log') 
ax.set_yscale('log')
#ax.set_xlim(P_min,P_max) 
#ax.set_ylim(Pdot_min,Pdot_max) 
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$L_{X}$ [erg s$^{-1}$]')

ax.plot( 
    t,
    L12_new,
    linestyle='-',
    linewidth=4,
    color=colors(1),
    rasterized=True,
)
ax.plot( 
    t,
    L13_new,
    linestyle='-',
    linewidth=4,
    color=colors(2),
    rasterized=True,
)
ax.plot( 
    t,
    L14_new,
    linestyle='-',
    linewidth=4,
    color=colors(3),
    rasterized=True,
)
ax.plot( 
    t,
    L15_new,
    linestyle='-',
    linewidth=4,
    color=colors(4),
    rasterized=True,
)
ax.plot( 
    t,
    L5e15_new,
    linestyle='-',
    linewidth=4,
    color=colors(5),
    rasterized=True,
)
plt.grid()

In [ ]:
# Create a table grid of magnetic field evolution.
B0 = np.array([1.e12, 1.e13, 1.e14, 1.e15, 5.e15])

L_t = np.vstack((L12_new, L13_new, L14_new, L15_new, L5e15_new)).T
print(t.shape)
print(L_t.shape)

Lx_interpolator = interpolate.RectBivariateSpline(
    t, 
    B0, 
    L_t, 
    bbox=[0.0, 1.e8, 1.e9, 1.e17], 
    kx=1, 
    ky=1
)

In [ ]:
#Pickle, unpickle and then plot again
with open('../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/interpolator_Lx.pkl', 'wb') as f:
    pickle.dump(Lx_interpolator, f)
with open('../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/interpolator_Lx.pkl', 'rb') as f:
    Lx_interpolator_import = pickle.load(f)

In [ ]:
t_eval = np.logspace(0., 6., 500)
B0_eval = np.logspace(9., 17, 30)

t_eval = np.array([1.e3, 1.e2])
B0_eval = np.logspace(13., 17, 2)

Lt_interp = Lx_interpolator_import.ev(t_eval, B0_eval)
print(Lt_interp.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(B0_eval))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale('log') 
ax.set_yscale('log')
#ax.set_xlim(P_min,P_max) 
#ax.set_ylim(Pdot_min,Pdot_max) 
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$L_{X}$ [erg s$^{-1}$]')

ax.plot( 
    t,
    L12_new,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
ax.plot( 
    t,
    L13_new,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
ax.plot( 
    t,
    L14_new,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
ax.plot( 
    t,
    L15_new,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
ax.plot( 
    t,
    L5e15_new,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)

for i in range(len(B0_eval)):
    ax.plot( 
        t_eval,
        Lt_interp[:,i],
        linestyle='-',
        linewidth=4,
        color=colors(i),
        rasterized=True,
        alpha=0.5
    )
plt.grid()

In [ ]:
sigma_SB = 5.67e-5
M_NS = 1.4*const.M_SUN
R_NS = 1.1e6
elambda = (1 - (2*const.G*M_NS)/(const.C**2 * R_NS))**(1./2.)
print(elambda)

In [ ]:
def T_from_Lx(Lx: np.ndarray) -> np.ndarray:
    """
    Calculating the temperature seen by an observer at infinity from the X-ray luminosity. 

    Args:
        Lx (np.ndarray): X-ray luminosity in erg/s.

    Returns:
        (np.ndarray): temperature seen by an observer at infinity in K.
    """
    R_inf = R_NS / elambda
    T_inf = (Lx / (4*np.pi*R_inf**2 * sigma_SB))**(1./4.)

    return T_inf

In [ ]:
T12_new = T_from_Lx(L12_new)
T13_new = T_from_Lx(L13_new)
T14_new = T_from_Lx(L14_new)
T15_new = T_from_Lx(L15_new)
T5e15_new = T_from_Lx(L5e15_new)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(log_B_range))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale('log') 
ax.set_yscale('log')
#ax.set_xlim(P_min,P_max) 
#ax.set_ylim(Pdot_min,Pdot_max) 
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$T_{\infty}$ [K]')

ax.plot( 
    t,
    T12_new,
    linestyle='-',
    linewidth=4,
    color=colors(1),
    rasterized=True,
    label=r"$B_0 = 10^{12}$ G",
)
ax.plot( 
    t,
    T13_new,
    linestyle='-',
    linewidth=4,
    color=colors(2),
    rasterized=True,
    label=r"$B_0 = 10^{13}$ G",
)
ax.plot( 
    t,
    T14_new,
    linestyle='-',
    linewidth=4,
    color=colors(3),
    rasterized=True,
    label=r"$B_0 = 10^{14}$ G",
)
ax.plot( 
    t,
    T15_new,
    linestyle='-',
    linewidth=4,
    color=colors(4),
    rasterized=True,
    label=r"$B_0 = 10^{15}$ G",
)
ax.plot( 
    t,
    T5e15_new,
    linestyle='-',
    linewidth=4,
    color=colors(5),
    rasterized=True,
    label=r"$B_0 = 5 \times 10^{15}$ G",
)
plt.legend(frameon=False, loc=0)
plt.grid()